Əvvəl notebooku və SQL bağlantısını hazırlayırıq.

In [2]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text, Unicode

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
from db import get_engine

engine = get_engine()

with engine.connect() as conn:
    db_name = conn.execute(text("SELECT DB_NAME()")).scalar()

print("Python və SQL Server hazırdır:", db_name)

Python və SQL Server hazırdır: BankAnalyticsDB


Kredit məlumatlarını oxuyaq

In [3]:
# Bronze-dakı birləşdirilmiş kredit məlumatlarını oxuyuruq
with engine.connect() as conn:
    silver_loans_df = pd.read_sql(
        text("SELECT * FROM bronze.loans_enriched"),
        conn
    )

In [4]:
silver_loans_df.head()

,loan_id,customer_id,application_date,product_type,amount,term_months,interest_rate,status,rejection_reason,defaulted,...,merge_date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday,time,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,L000001,C03267,1/1/2026,Biznes krediti,18174.08,34,11.34,Təsdiqləndi,NaN,0.0,...,2026-01-01,1,Yeni il,0,0,0,2026-01-01,5.5,0.0,18.6
1,L000002,C00082,1/2/2026,İstehlak krediti,14942.94,15,13.74,Rədd edildi,Yüksək borc yükü,NaN,...,2026-01-02,1,Yeni il,0,0,0,2026-01-02,5.2,12.9,22.6
2,L000003,C00715,1/2/2026,İstehlak krediti,24217.34,58,14.09,Təsdiqləndi,NaN,0.0,...,2026-01-02,1,Yeni il,0,0,0,2026-01-02,2.9,5.3,36.6
3,L000004,C02969,1/2/2026,Biznes krediti,14890.76,58,14.82,Rədd edildi,Aşağı kredit skoru,NaN,...,2026-01-02,1,Yeni il,0,0,0,2026-01-02,2.0,0.7,25.5
4,L000005,C00178,1/3/2026,İstehlak krediti,6382.58,21,13.19,Təsdiqləndi,NaN,0.0,...,2026-01-03,0,NaN,1,0,0,2026-01-03,3.6,0.0,10.7


In [5]:
silver_loans_df.shape

(4820, 32)

In [6]:
silver_loans_df.duplicated().sum()

np.int64(15)

In [7]:
silver_loans_df = silver_loans_df.drop_duplicates()

In [8]:
silver_loans_df.duplicated().sum()

np.int64(0)

In [9]:
silver_loans_df.isna().sum()

loan_id                   0
customer_id               0
application_date          0
product_type              0
amount                    0
term_months               0
interest_rate            50
status                    0
rejection_reason       3070
defaulted              1826
age                       5
gender                    5
city                      5
branch                    5
tenure_years              5
credit_score              5
balance                   5
salary                    5
products_count            5
has_card                  5
is_active                 5
churned                   5
merge_date                0
is_holiday                0
holiday_name           4768
is_weekend                0
is_workday                0
is_pre_holiday            0
time                      5
temperature_2m_mean       5
precipitation_sum         5
wind_speed_10m_max        5
dtype: int64

In [10]:
silver_loans_df.groupby("status")["defaulted"].apply(lambda x: x.isna().sum())

status
Rədd edildi    1777
Təsdiqləndi       0
rədd edildi      49
təsdiqləndi       0
Name: defaulted, dtype: int64

defaulted kredit gecikmesini gosterir. ve 1826 bos deyerin hamisi redd edilen kreditlere aiddir. bu normaldir. 

Status yazılışlarını eyniləşdirək

In [11]:
silver_loans_df["status"] = silver_loans_df["status"].str.lower()

In [12]:
silver_loans_df["status"].value_counts()

status
təsdiqləndi    2979
rədd edildi    1826
Name: count, dtype: int64

interest rate de bos olan 50 deyere baxaq


In [13]:
silver_loans_df[silver_loans_df["interest_rate"].isna()]["status"].value_counts(dropna=False)

status
təsdiqləndi    34
rədd edildi    16
Name: count, dtype: int64

34-ü təsdiqlənib — bunlarda faiz dərəcəsi olmalıdır.
16-sı rədd edilib — bunlarda faizin boş qalması normal ola bilər.

kredit novlerinin medianlarini tapaq ve bu deyerlere uygun bosluqlari dolduraq

In [14]:
silver_loans_df.groupby("product_type")["interest_rate"].median()

product_type
Avtomobil krediti    11.00
Biznes krediti       12.13
İpoteka               8.09
İstehlak krediti     14.06
Name: interest_rate, dtype: float64

In [15]:
mask = silver_loans_df["status"] == "təsdiqləndi"
medians = silver_loans_df.groupby("product_type")["interest_rate"].transform("median")

silver_loans_df.loc[mask, "interest_rate"] = silver_loans_df.loc[mask, "interest_rate"].fillna(medians)

yerde qalan bos deyerlerin sayina baxaq

In [16]:
silver_loans_df["interest_rate"].isna().sum()

np.int64(16)

novbeti sutunlarin bos deyerlerine baxaq

In [17]:
silver_loans_df[silver_loans_df["age"].isna()][
    ["loan_id", "customer_id", "age", "gender", "city", "salary"]
]

,loan_id,customer_id,age,gender,city,salary
1402,L001398,C09999,NaN,NaN,NaN,NaN
1754,L001749,C09999,NaN,NaN,NaN,NaN
2672,L002665,C09999,NaN,NaN,NaN,NaN
3698,L003687,C09999,NaN,NaN,NaN,NaN
3795,L003784,C09999,NaN,NaN,NaN,NaN


In [18]:
pd.read_sql(
    "SELECT * FROM bronze.customers WHERE customer_id = 'C09999'",
    engine
)

,customer_id,age,gender,city,branch,tenure_years,credit_score,balance,salary,products_count,has_card,is_active,churned


yalniz bir musterinin adi customer cedvelinde yoxdu

baxaq ki bu musterinin tesdiqlenmis krediti varmi?

In [19]:
silver_loans_df[silver_loans_df["customer_id"] == "C09999"]["status"].value_counts()

status
təsdiqləndi    4
rədd edildi    1
Name: count, dtype: int64

bu 5 krediti silməyək. Çünki kreditlərin öz məlumatları mövcuddur, sadəcə müştəri məlumatları çatışmır. Silmək kredit sayını və ümumi məbləği səhv göstərə bilər.

In [20]:
silver_loans_df["customer_missing"] = silver_loans_df["age"].isna()

silver_loans_df["customer_missing"].value_counts()

customer_missing
False    4800
True        5
Name: count, dtype: int64

In [21]:
silver_loans_df.isna().sum()[lambda x: x > 0]

interest_rate            16
rejection_reason       3070
defaulted              1826
age                       5
gender                    5
city                      5
branch                    5
tenure_years              5
credit_score              5
balance                   5
salary                    5
products_count            5
has_card                  5
is_active                 5
churned                   5
holiday_name           4768
time                      5
temperature_2m_mean       5
precipitation_sum         5
wind_speed_10m_max        5
dtype: int64

In [22]:
silver_loans_df.groupby("is_holiday")["holiday_name"].apply(lambda x: x.isna().sum())

is_holiday
0    4768
1       0
Name: holiday_name, dtype: int64

bayram gunlerinde bos deyer yoxdur

In [23]:
silver_loans_df.shape


(4805, 33)

indi bu cedveli sql e yazaq

In [24]:
silver_loans_df.to_sql(
    "loans", engine, schema="silver",
    if_exists="replace", index=False,
    dtype={c: Unicode(255) for c in silver_loans_df.select_dtypes(include=["object", "string"]).columns if c != "merge_date"}
)

17

Bronze dan transaction enriched faylini oxuyaq

In [25]:
# Bronze-dakı birləşdirilmiş kredit məlumatlarını oxuyuruq
with engine.connect() as conn:
    silver_transactions_df = pd.read_sql(
        text("SELECT * FROM bronze.transactions_enriched"),
        conn
    )
   

In [26]:
silver_transactions_df.head()

,transaction_id,customer_id,date,time,amount,currency,category,channel,age,gender,...,holiday_name,is_weekend,is_workday,is_pre_holiday,rate_currency,rate_base,rate,temperature_2m_mean,precipitation_sum,wind_speed_10m_max
0,T0000001,C02755,1/1/2026,7:03:37,21.05,AZN,Nağdlaşdırma,ATM,67,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
1,T0000002,C01051,1/1/2026,7:11:06,23.04,AZN,Nağdlaşdırma,ATM,41,Qadın,...,Yeni il,0,0,0,NaN,NaN,NaN,3.5,0.0,14.8
2,T0000003,C03032,1/1/2026,7:11:29,54.70,AZN,Köçürmə,Mobil tətbiq,36,Qadın,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
3,T0000004,C04099,1/1/2026,7:19:02,18.16,AZN,Mobil ödəniş,Mobil tətbiq,25,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,4.1,0.0,28.7
4,T0000005,C02686,1/1/2026,7:19:11,70.54,AZN,Nağdlaşdırma,ATM,44,Kişi,...,Yeni il,0,0,0,NaN,NaN,NaN,7.2,0.3,10.8


In [27]:
silver_transactions_df.shape

(77396, 32)

In [28]:
silver_transactions_df.duplicated().sum()

np.int64(0)

In [29]:
silver_transactions_df.isna().sum()

transaction_id             0
customer_id                0
date                       0
time                       0
amount                     0
currency                   0
category                   0
channel                    0
age                        0
gender                     0
city                       0
branch                     0
tenure_years               0
credit_score               0
balance                    0
salary                     0
products_count             0
has_card                   0
is_active                  0
churned                    0
merge_date                 0
is_holiday                 0
holiday_name           68014
is_weekend                 0
is_workday                 0
is_pre_holiday             0
rate_currency          72100
rate_base              72100
rate                   72100
temperature_2m_mean        0
precipitation_sum          0
wind_speed_10m_max         0
dtype: int64

In [30]:
silver_transactions_df.groupby("currency")["rate"].apply(
    lambda x: x.isna().sum()
)

currency
AZN    72100
EUR        0
USD        0
Name: rate, dtype: int64

Bu normaldır. Çünki valyuta məzənnəsi cədvəlimizdə USD və EUR üçün məzənnələr var, AZN üçün isə yoxdur.

AZN-in AZN-ə məzənnəsi həmişə 1-dir. Ona görə bu boşluqları 1 ilə doldura bilərik.

In [31]:
mask = silver_transactions_df["currency"] == "AZN"
silver_transactions_df.loc[mask, "rate"] = 1

In [32]:
silver_transactions_df["rate"].isna().sum()

np.int64(0)

In [33]:
silver_transactions_df.groupby("is_holiday")["holiday_name"].apply(
    lambda x: x.isna().sum())

is_holiday
0    68014
1        0
Name: holiday_name, dtype: int64

bayram gunlerinde bos deyerler yoxdur

In [34]:
silver_transactions_df.isna().sum()[lambda x: x > 0]

holiday_name     68014
rate_currency    72100
rate_base        72100
dtype: int64

In [35]:
mask = silver_transactions_df["currency"] == "AZN"

silver_transactions_df.loc[mask, "rate_currency"] = "AZN"
silver_transactions_df.loc[mask, "rate_base"] = "AZN"

silver_transactions_df.isna().sum()[lambda x: x > 0]

holiday_name    68014
dtype: int64

In [36]:
silver_transactions_df["amount"].describe()

count    77396.000000
mean        88.026543
std        149.161604
min          0.380000
25%         21.650000
50%         45.375000
75%         98.420000
max       9652.100000
Name: amount, dtype: float64

maksimum əməliyyat meblegi orta meblegden cox boyukdur bunu arasdiraq ən boyuk 10 emeliyyatin hansi novune aid oldugunu tapaq

In [37]:
silver_transactions_df.nlargest(10, "amount")[
    ["transaction_id", "amount", "currency", "category"]
]

,transaction_id,amount,currency,category
11745,T0011746,9652.10,AZN,Depozit
3294,T0003295,5132.73,AZN,Depozit
48910,T0048911,3731.84,AZN,Depozit
23107,T0023108,3318.02,AZN,Depozit
46048,T0046049,3251.40,AZN,Depozit
69814,T0069815,3249.10,AZN,Depozit
70342,T0070343,3218.01,AZN,Depozit
72570,T0072571,3189.50,AZN,Depozit
69832,T0069833,3134.51,AZN,Depozit
23493,T0023494,3070.16,AZN,Depozit


ən yüksək əməliyyatlar depozit emeliyyatıdır bu da  normaldır

indi ise kategoriyalar üzrə araşdıraq

In [38]:
silver_transactions_df.groupby("category")["amount"].agg(
    ["count", "mean", "max"]
).round(2)

,count,mean,max
category,,,
Alış-veriş,26453,41.03,1621.76
Depozit,1133,679.69,9652.10
Kredit ödənişi,2256,247.64,2058.96
Köçürmə,14762,134.81,1913.51
Mobil ödəniş,8342,22.54,349.74
Nağdlaşdırma,14337,123.15,1661.83
Ödəniş,10113,45.01,581.81


tranzaksiya eden musterilerden necesinin kredit cedvelinde adi olmamasini arasdiraq

In [39]:
silver_transactions_df[
    ~silver_transactions_df["customer_id"].isin(
        silver_loans_df["customer_id"]
    )
]["customer_id"].nunique()

2075

demeli 2075 müştəri tranzaksiya edir amma kredit goturmeyib


In [42]:
customers_df = pd.read_sql(
    "SELECT * FROM bronze.customers",
    engine
)


bankda nəqədər müştərinin qaldığını ve ne qədərinin tərk etdiyini araşdıraq

In [41]:
churn_summary = customers_df.groupby("churned").size().reset_index(name="say")

churn_summary["faiz"] = (churn_summary["say"] / 5000 * 100).round(2)

churn_summary

,churned,say,faiz
0,0,3992,79.84
1,1,1008,20.16


sql golda göndərək

In [43]:
churn_summary.to_sql(
    "churn_summary",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

print("Gold cədvəli uğurla yazıldı!")

Gold cədvəli uğurla yazıldı!


In [44]:
customers_df.groupby("is_active")["churned"].agg(
    ["count", "sum", "mean"]
).round(3)

,count,sum,mean
is_active,,,
0,2029,584,0.288
1,2971,424,0.143


In [45]:
aktivlik = customers_df.groupby("is_active")["churned"].agg(
    ["count", "sum", "mean"]
).reset_index()

aktivlik.columns = [
    "Aktivlik",
    "Müştəri sayı",
    "Bankı tərk edənlər",
    "Churn faizi"
]

aktivlik["Aktivlik"] = aktivlik["Aktivlik"].map({
    0: "Aktiv deyil",
    1: "Aktivdir"
})

aktivlik["Churn faizi"] = (aktivlik["Churn faizi"] * 100).round(2)

aktivlik

,Aktivlik,Müştəri sayı,Bankı tərk edənlər,Churn faizi
0,Aktiv deyil,2029,584,28.78
1,Aktivdir,2971,424,14.27


In [47]:
aktivlik.to_sql(
    "churn_by_activity",
    engine,
    schema="gold",
    if_exists="replace",
    index=False,
dtype={"Aktivlik": Unicode(255)}
)

print("Cədvəl Gold-a yazıldı!")

Cədvəl Gold-a yazıldı!


İndi churn-un başqa hansı amillərlə əlaqəli olduğunu araşdıraq:
1. Məhsul sayı — az bank məhsulu istifadə edənlər daha çox gedirmi?

In [48]:
mehsul = customers_df.groupby("products_count")["churned"].agg(
    ["count", "sum", "mean"]
).reset_index()

mehsul.columns = [
    "Məhsul sayı",
    "Müştəri sayı",
    "Bankı tərk edənlər",
    "Churn faizi"
]

mehsul["Churn faizi"] = (mehsul["Churn faizi"] * 100).round(2)

mehsul

,Məhsul sayı,Müştəri sayı,Bankı tərk edənlər,Churn faizi
0,1,2289,520,22.72
1,2,2114,214,10.12
2,3,441,171,38.78
3,4,156,103,66.03


4 mehsulu olan musterilerin getme faizi daha coxdur

golda yazaq

In [49]:
mehsul.to_sql(
    "churn_by_products",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

print("Cədvəl Gold-a yazıldı!")

Cədvəl Gold-a yazıldı!


In [52]:
mehsul_aktivlik = customers_df.groupby(
    ["products_count", "is_active"]
)["churned"].agg(
    say="count",
    gedən="sum",
    churn_faizi="mean"
).assign(
    churn_faizi=lambda x: (x["churn_faizi"] * 100).round(2)
)
mehsul_aktivlik 


say  gedən  churn_faizi
products_count is_active                          
1              0          1037    328        31.63
               1          1252    192        15.34
2              0           787    133        16.90
               1          1327     81         6.10
3              0           146     75        51.37
               1           295     96        32.54
4              0            59     48        81.36
               1            97     55        56.70

Aktiv olmayanlar: 59 nəfərdən 48-i bankı tərk edib — 81.36%



Aktiv olanlar: 97 nəfərdən 55-i bankı tərk edib — 56.70%

sql golda yazaq

In [53]:
mehsul_aktivlik.to_sql(
    "churn_products_activity",
    engine,
    schema="gold",
    if_exists="replace",
    index=False
)

8